<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_30_Windows_Registry_USB_Device_History_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# Experiment 2: Windows Registry USB Device History Analyzer
# ==========================================================

from datetime import datetime


# Function to parse USBSTOR registry data
def parse_usbstor(registry_data):
    """
    Convert USBSTOR registry dictionary into a list of devices
    sorted by the most recent connection time.
    """

    devices = []

    for device_id, info in registry_data.items():
        entry = {
            "device_id": device_id
        }

        entry.update(info)
        devices.append(entry)

    devices.sort(
        key=lambda d: datetime.strptime(
            d["last_connected"],
            "%Y-%m-%d %H:%M:%S"
        ),
        reverse=True
    )

    return devices


# Function to find USB devices connected near a target time
def find_device_near_time(devices, target_time_str, window_minutes=30):
    """
    Return devices whose last_connected timestamp
    falls within the specified time window.
    """

    target = datetime.strptime(
        target_time_str,
        "%Y-%m-%d %H:%M:%S"
    )

    matches = []

    for device in devices:

        last = datetime.strptime(
            device["last_connected"],
            "%Y-%m-%d %H:%M:%S"
        )

        delta_minutes = abs(
            (target - last).total_seconds()
        ) / 60

        if delta_minutes <= window_minutes:
            matches.append(device)

    return matches


# ==========================================================
# Test Cases
# ==========================================================

def test_experiment2():

    registry_data = {

        "USB\\VID_0781&PID_5567\\4C531001234": {
            "serial": "4C531001234",
            "friendly_name": "SanDisk Cruzer Blade",
            "first_connected": "2025-11-01 09:00:00",
            "last_connected": "2026-02-10 17:42:00",
        },

        "USB\\VID_090C&PID_1000\\A1002233": {
            "serial": "A1002233",
            "friendly_name": "Kingston DataTraveler",
            "first_connected": "2024-06-01 10:00:00",
            "last_connected": "2024-06-01 10:15:00",
        }

    }

    # Parse registry data
    devices = parse_usbstor(registry_data)

    print("USB Device History (Most Recent First):")
    print("---------------------------------------")

    for device in devices:
        print(f"Device Name      : {device['friendly_name']}")
        print(f"Device ID        : {device['device_id']}")
        print(f"Serial Number    : {device['serial']}")
        print(f"First Connected  : {device['first_connected']}")
        print(f"Last Connected   : {device['last_connected']}")
        print()

    # Verify sorting
    assert devices[0]["friendly_name"] == "SanDisk Cruzer Blade"

    # Search near the suspected exfiltration time
    matches = find_device_near_time(
        devices,
        "2026-02-10 17:35:00",
        window_minutes=30
    )

    print("Matching USB Devices:")
    print("---------------------")

    for device in matches:
        print(device["friendly_name"])

    matched_names = [m["friendly_name"] for m in matches]

    assert "SanDisk Cruzer Blade" in matched_names
    assert "Kingston DataTraveler" not in matched_names

    print("\nAll test cases passed.")


# Run the test
test_experiment2()

USB Device History (Most Recent First):
---------------------------------------
Device Name      : SanDisk Cruzer Blade
Device ID        : USB\VID_0781&PID_5567\4C531001234
Serial Number    : 4C531001234
First Connected  : 2025-11-01 09:00:00
Last Connected   : 2026-02-10 17:42:00

Device Name      : Kingston DataTraveler
Device ID        : USB\VID_090C&PID_1000\A1002233
Serial Number    : A1002233
First Connected  : 2024-06-01 10:00:00
Last Connected   : 2024-06-01 10:15:00

Matching USB Devices:
---------------------
SanDisk Cruzer Blade

All test cases passed.
